<a href="https://colab.research.google.com/github/Caio-Oliveira98/PYTHON/blob/main/Classe_DocumentProcessor_Robusta_e_Endpoints.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import json
from PIL import Image
import pytesseract
import numpy as np
import cv2 # Biblioteca essencial para pré-processamento de Visão Computacional

# --- Configurações ---
# Nome do arquivo de padrões de validação. DEVE ESTAR NO MESMO DIRETÓRIO.
VALIDATION_FILEPATH = 'validation_patterns.json'
# Configuração (Ajuste o caminho do executável do tesseract se necessário)
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'


class DocumentProcessor:
    """
    Estrutura de classe especializada para processamento de documentos (imagem/PDF) com foco
    em OCR robusto (pré-processamento) e validação flexível (padrões externos).

    Métodos Chave:
    - _load_document_from_path: Carrega de um caminho de arquivo.
    - _load_document_from_matrix: Carrega de matriz em memória (OpenCV/PIL).
    - preprocess_image: Aplica a robustez (binarização, deskewing).
    - apply_ocr_with_positions: Executa o OCR e retorna texto com coordenadas.
    - validate_pattern: Valida o texto extraído contra padrões RegEx carregados do JSON.
    """

    def __init__(self, tesseract_cmd=None, lang='por'):
        """
        Inicializa o processador, configurando o Tesseract e carregando os padrões de validação.
        """
        if tesseract_cmd:
            pytesseract.pytesseract.tesseract_cmd = tesseract_cmd
        self.lang = lang
        self.patterns = self._load_validation_patterns()

    def _load_validation_patterns(self) -> dict:
        """
        Carrega os padrões de RegEx de um arquivo JSON externo.
        Isso garante o desacoplamento das regras de negócio (alta flexibilidade).
        """
        try:
            with open(VALIDATION_FILEPATH, 'r', encoding='utf-8') as f:
                return json.load(f)
        except FileNotFoundError:
            print(f"AVISO: Arquivo de padrões '{VALIDATION_FILEPATH}' não encontrado. Usando padrões vazios.")
            return {}
        except json.JSONDecodeError:
            print("ERRO: O arquivo JSON de padrões está mal formatado.")
            return {}

    def _load_document_from_path(self, file_path: str) -> np.ndarray:
        """
        Função para abrir um arquivo a partir de um caminho, retornando
        uma matriz NumPy (np.ndarray), que é o formato exigido pelo OpenCV.
        """
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Arquivo não encontrado: {file_path}")

        img_np = cv2.imread(file_path)
        if img_np is None:
            raise ValueError("Não foi possível carregar a imagem ou formato não suportado.")

        return img_np

    def _load_document_from_matrix(self, img_data: np.ndarray or Image.Image) -> np.ndarray:
        """
        Função que recebe dados de imagem em memória (matriz OpenCV ou PIL Image)
        e garante que a saída seja uma matriz NumPy (np.ndarray) para o pré-processamento.
        """
        if isinstance(img_data, Image.Image):
            # Converte PIL Image (RGB) para NumPy (BGR - padrão OpenCV)
            img_np = np.array(img_data)
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
            return img_np

        elif isinstance(img_data, np.ndarray):
            return img_data

        else:
            raise TypeError("O dado de entrada deve ser uma matriz NumPy ou um objeto PIL.Image.")

    def preprocess_image(self, img_np: np.ndarray) -> Image.Image:
        """
        Módulo de Robustez: Aplica pré-processamento na matriz da imagem antes do OCR.
        1. Binarização: Aumenta o contraste entre fundo e texto.
        2. Deskewing (Correção de Inclinação): Alinha o documento.
        """
        # 1. Converte para tons de cinza
        gray = cv2.cvtColor(img_np, cv2.COLOR_BGR2GRAY)

        # 2. Binarização usando Otsu (ideal para documentos)
        _, binarized = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # 3. Deskewing (Correção de inclinação)
        coords = np.column_stack(np.where(binarized < 255))
        # Calcula o ângulo de rotação
        angle = cv2.minAreaRect(coords)[-1]

        if angle < -45:
            angle = -(90 + angle)
        else:
            angle = -angle

        # Aplica a rotação se o ângulo for relevante
        if abs(angle) > 1.0:
            (h, w) = binarized.shape[:2]
            center = (w // 2, h // 2)
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            deskewed = cv2.warpAffine(binarized, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
        else:
            deskewed = binarized

        # 4. Converte a matriz processada para o formato PIL para o Tesseract
        return Image.fromarray(deskewed)

    def apply_ocr_with_positions(self, image_np: np.ndarray) -> dict:
        """
        Aplica o pré-processamento e o OCR, retornando o texto completo e as posições (bounding boxes).
        """
        # A robustez é aplicada primeiro
        preprocessed_image = self.preprocess_image(image_np)

        try:
            # Aplica OCR com Tesseract, extraindo texto e coordenadas
            data = pytesseract.image_to_data(preprocessed_image, lang=self.lang, output_type=pytesseract.Output.DICT)

            structured_data = []

            # Filtra e estrutura os dados
            for i in range(len(data['text'])):
                text = data['text'][i].strip()
                if text and data['conf'][i] > 0:
                    structured_data.append({
                        'text': text,
                        'confidence': data['conf'][i],
                        'box': {
                            'left': data['left'][i],
                            'top': data['top'][i],
                            'width': data['width'][i],
                            'height': data['height'][i]
                        }
                    })

            # Extrai o texto completo para validação
            full_text = pytesseract.image_to_string(preprocessed_image, lang=self.lang).strip()

            return {
                "full_text": full_text,
                "words_with_positions": structured_data
            }

        except Exception as e:
            return {"error": f"Erro no processo OCR: {e}"}

    def validate_pattern(self, full_text: str, pattern_type: str, custom_pattern: str = None) -> dict:
        """
        Rotina para validar se uma entrada de texto corresponde a um padrão,
        buscando o RegEx no dicionário carregado de 'validation_patterns.json'.
        """

        # Prioriza o padrão customizado se fornecido, senão busca o tipo (ex: NUMERO_PASSAPORTE)
        pattern_to_use = custom_pattern if custom_pattern else self.patterns.get(pattern_type.upper())

        if not pattern_to_use:
            return {"pattern_found": False, "match": None, "validation_status": "Padrão de validação não reconhecido/carregado."}

        try:
            match = re.search(pattern_to_use, full_text)

            if match:
                return {
                    "pattern_found": True,
                    "match": match.group(0),
                    "validation_status": f"Padrão '{pattern_type}' encontrado e validado."
                }
            else:
                return {
                    "pattern_found": False,
                    "match": None,
                    "validation_status": f"Padrão '{pattern_type}' não encontrado."
                }
        except re.error as e:
            return {"pattern_found": False, "match": None, "validation_status": f"Erro de expressão regular: {e}"}

# --- Lista de Endpoints Sugeridos (Requisito do Entregável) ---
"""
LISTA DE ENDPOINTS SUGERIDOS PARA O BACK-END (YouVisa OCR Module):

Estes endpoints expõem a funcionalidade da classe DocumentProcessor como um serviço REST.

1. POST /processar_documento_file
   - Método: POST
   - Descrição: Recebe um arquivo (multipart/form-data) para processamento completo.
   - Entrada (Body): file (documento), pattern_type (string).
   - Saída (JSON): Texto completo, posições e status de validação.

2. POST /processar_documento_base64
   - Método: POST
   - Descrição: Recebe uma imagem codificada em Base64 (matriz em memória) para processamento.
   - Entrada (Body): base64_image (string), pattern_type (string).
   - Saída (JSON): Texto completo, posições e status de validação.

3. GET /status_saude
   - Método: GET
   - Descrição: Health check para monitoramento.
   - Saída (JSON): Status do serviço e confirmação de carregamento dos padrões.
"""

if __name__ == '__main__':
    # --- DEMONSTRAÇÃO DE USO (Simulação do Backend em Ação) ---
    print("\n--- INICIALIZANDO DOCUMENT PROCESSOR ---")

    processor = DocumentProcessor(lang='por')

    # 1. Criação de uma imagem mock em memória (simulando documento inclinado e com dados)
    mock_img_data = np.zeros((300, 600, 3), dtype=np.uint8) + 255
    text_to_insert = "DATA: 15/12/2023. Passaporte: AX1234567. Nome: J. S. N. Viana."
    cv2.putText(mock_img_data, text_to_insert, (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)

    # Simula uma inclinação de 5 graus para testar o Deskewing
    (h, w) = mock_img_data.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, 5, 1.0)
    mock_img_data = cv2.warpAffine(mock_img_data, M, (w, h))

    # 2. Carregar a matriz (Simula a entrada de dados via API)
    image_to_process = processor._load_document_from_matrix(mock_img_data)

    # 3. Aplicar OCR e Validação de Passaporte e Data
    print("\n--- PROCESSAMENTO OCR COM PRÉ-PROCESSAMENTO ---")
    ocr_result = processor.apply_ocr_with_positions(image_to_process)

    val_passaporte = processor.validate_pattern(ocr_result["full_text"], "NUMERO_PASSAPORTE")
    val_data = processor.validate_pattern(ocr_result["full_text"], "DATA_DD_MM_AAAA")

    print("\n--- RELATÓRIO DE RESULTADO DA YOUVISA ---")
    print(f"Texto Completo Extraído:\n'{ocr_result['full_text'].replace('\n', ' ')}'")

    print("\n[Relatório de Validação de Padrões]:")
    print("Passaporte:", json.dumps(val_passaporte, indent=2))
    print("Data:", json.dumps(val_data, indent=2))

    print(f"\nTotal de Palavras Extraídas com Posição: {len(ocr_result['words_with_positions'])}")